In [2]:
#data generation
import pandas as pd
import numpy as np

np.random.seed(72) #set seed for reproducability
n = 200 #trials

exp_var = np.random.uniform(low=2,high=10, size = n) #exp var
#print(exp)

exp_var_2 = 0.5 * exp_var + np.random.normal(3, 0.5, n)

#make lin reg
beta_0, beta_1, beta_2 = 4, 5, 1.5
initial_response = beta_1 + (beta_2*exp_var) + (beta_2 * exp_var_2)

#residuals
residuals = np.random.normal(loc = 0, scale = 1.2, size= n)
response = initial_response + residuals
#adding numpy arrays, adds the correct indexes, and for missing values, it just i
#print(response)


df = pd.DataFrame({'explanatory var 1': exp_var,
                   'explanatory var 2': exp_var_2,
                   'response var': response
                   })
df.head()


,explanatory var 1,explanatory var 2,response var
0,2.853906,4.614877,16.049482
1,7.474741,6.991428,26.293937
2,6.279701,6.300166,23.444636
3,4.953497,5.837439,20.050923
4,5.300917,5.323199,20.746840


In [3]:
#data removal - mcar - only do for response for convenience
#coin flip like 20%
#have to make new column called missingness indicator

mcar_df = df.copy()
np.random.seed(72)
mcar_df['indicator'] = np.random.uniform(0, 10, size=len(df))
#could use a bernoulli instead
mcar_df.loc[(mcar_df['indicator']<1.5),['response var']] = None

display(mcar_df.head(10))


#can we use df.loc?


,explanatory var 1,explanatory var 2,response var,indicator
0,2.853906,4.614877,NaN,1.067382
1,7.474741,6.991428,26.293937,6.843426
2,6.279701,6.300166,23.444636,5.349626
3,4.953497,5.837439,20.050923,3.691871
4,5.300917,5.323199,20.746840,4.126146
5,6.702780,6.573294,23.877211,5.878475
6,7.728590,7.705092,28.784335,7.160738
7,3.386672,4.544854,19.789953,1.733340
8,2.533938,3.869837,NaN,0.667423
9,6.005431,6.470642,22.760650,5.006789


In [4]:
#removal - mar
#higher values of exp var 2
mar_df = df.copy()
# make the exp var 2 value a proportion, then scale it out of 1, plot on sigmond to get probabilities
mar_df['z'] = (exp_var_2 - exp_var_2.mean()) / exp_var_2.std()
#this runs a new seed every time?
np.random.seed(72)

mar_df['sigmoid'] = 1 / (1 + np.exp(-2 * mar_df["z"]))
mar_df["missing"] = np.random.rand(len(mar_df)) < mar_df["sigmoid"]
mar_df.loc[mar_df['missing']==True,'response var']=None
#use .loc
#apply smae method for
#not sure if this sigmoid is really necessary

mar_df.head()
#use a sigmoid function
#same principle generate the indicator column, but only for certain values of the exp var 2 or whatever

#do this tonight

,explanatory var 1,explanatory var 2,response var,z,sigmoid,missing
0,2.853906,4.614877,NaN,-1.050379,0.109023,True
1,7.474741,6.991428,NaN,0.861500,0.848515,True
2,6.279701,6.300166,NaN,0.305395,0.648121,True
3,4.953497,5.837439,NaN,-0.066858,0.466621,True
4,5.300917,5.323199,20.74684,-0.480551,0.276658,False


In [25]:
mnar_df = df.copy()
display(mnar_df.head())
mnar_df['z'] = (response - response.mean()) / response.std()
print(response.mean())
print(response.std())

np.random.seed(72)

mnar_df['sigmoid'] = 1 / (1 + np.exp(-2 * mnar_df["z"]))
mnar_df["missing"] = np.random.rand(len(mnar_df)) < mnar_df["sigmoid"]
display(mnar_df.head())
mnar_df.loc[mnar_df['missing']==True,'response var']=None
#use .loc
#removal - mnar
mnar_df.head()
#lower values of resp are more likely to be missing
#could also do this for the

,explanatory var 1,explanatory var 2,response var
0,2.853906,4.614877,16.049482
1,7.474741,6.991428,26.293937
2,6.279701,6.300166,23.444636
3,4.953497,5.837439,20.050923
4,5.300917,5.323199,20.746840


22.79054732099668
5.163853115907453


,explanatory var 1,explanatory var 2,response var,z,sigmoid,missing
0,2.853906,4.614877,16.049482,-1.305433,0.068442,False
1,7.474741,6.991428,26.293937,0.678445,0.795254,True
2,6.279701,6.300166,23.444636,0.126667,0.562997,True
3,4.953497,5.837439,20.050923,-0.530539,0.257104,False
4,5.300917,5.323199,20.746840,-0.395772,0.311837,False


,explanatory var 1,explanatory var 2,response var,z,sigmoid,missing
0,2.853906,4.614877,16.049482,-1.305433,0.068442,False
1,7.474741,6.991428,NaN,0.678445,0.795254,True
2,6.279701,6.300166,NaN,0.126667,0.562997,True
3,4.953497,5.837439,20.050923,-0.530539,0.257104,False
4,5.300917,5.323199,20.746840,-0.395772,0.311837,False


In [ ]:
#then we can run models like gmm, and knn on these
#also use methods for estimating beta parameters to assess accuracy